In [ ]:
from utilities import *
from init_sol_prep import init_prep
from HOIPM import set_ipm_params, HOIPM_FNS, output_process
from problems.elliptope import elliptope
from problems.EdK import EdK
from problems.problem_generator_source import GenSDODims, instance_loader, SDOGen
from problems.PDG import *
from problems.Hauenstein_ssc import Hauen_ssc
import numpy as np
from pathlib import Path

np.set_printoptions(edgeitems=30, linewidth=100000)

In [ ]:
problem = input("Enter the problem of choice (Options: 'elliptope', 'EdK', 'GenSDO', 'PDG', 'HSSC'): ")
print(f"Selected problem: {problem}")

# Call problem data
if problem == 'elliptope':
    n, m, A, b, C, X_init, y_init, S_init, Xopt, yopt, Sopt = elliptope()

elif problem == 'EdK':
    # Initial Solutions available for n = 3,4,5
    n = int(input("Enter the dimension of EdK problem (available dimensions 3,4, and 5):"))
    n, m, A, b, C, X_init, y_init, S_init, Xopt, yopt, Sopt = EdK(n)

elif problem == 'GenSDO':
    instance = input("Enter the instance:") 
    GenProblemsLib = Path("problems/GeneratedProblems").resolve()
    mat_files = [f.name for f in GenProblemsLib.glob("*.mat")]
 
    if instance+'.mat' in mat_files: 
        n, m, A, b, C, X_init, y_init, S_init, Xopt, yopt, Sopt = instance_loader(instance)
        print(f'{instance} loaded!')       
    else:
        n, n_B, n_N, m = map(int, input("Enter n-1, n_B, n_N-1, m-1 (exact order separated with space):").split())
        gen_sdo_dims = GenSDODims(n=n, n_B=n_B, n_N=n_N, m=m)
        Xopt,Sopt = -np.eye(n+1), -np.eye(n+1)
        while np.linalg.eigh(Xopt+Sopt)[0][0] < 0:
            (Q,A,b,C,Xopt,yopt,Sopt,X_init,y_init,S_init)= SDOGen(gen_sdo_dims)
        print(f'Problem generated with (n, n_B, n_N, n_T, m)={(n+1, n_B, n_N+1, n+1-n_B-n_N-1, m+1)}!')
    
elif problem == 'PDG':
   instance = input("Enter the name of instance:")
   nSelf, mSelf, A, b, C, X_init, y_init, S_init = PDG_HSDE(instance)
   n = nSelf - 1
   m = mSelf - 1 
   print(f"Uploaded file: {instance}")

elif problem == 'HSSC':
    n, m, A, b, C, X_init, y_init, S_init = Hauen_ssc()

# Prepare the initial solution
X0, y0, S0 = init_prep(n, m, A,b,C, X_init, y_init, S_init)

print('Initial solution ready! Proceed to solve!')

In [ ]:
# Algorithm parameters
HOIPMParams = set_ipm_params(beta=0.5, cent_tol=1e-1, 
                             crawl_step=0.9999, 
                             alpha=1, 
                             precision=1e-10, 
                             norm_thrsh=1e-16, cent_thrsh=1e-12, feas_tol=1e-10, 
                             reduction=1-14, 
                             gamma=1.05)

params = [] # Pairs of (ρ, p)
for i in [1, 2]:
  for j in [i, 2*i, 3*i]:
    params.append((i, j))

# Output Dictionaries
muRecs  = {}
Records = {}
orders_used = {}
Drv_norm = {}

# Running the Algorithm
for rho, p in params: 
  print(f'###################### (ρ, p)={rho,p} ######################')
  (XX,yy,SS,mu_reached,muRecs[(rho,p)],Records[(rho,p)], orders_used[(rho,p)], Drv_norm[(rho,p)]) = HOIPM_FNS(A,b,C,m+1,n+1,X0,y0,S0,
                                                                                                              rho,p,
                                                                                                              HOIPMParams);
  if np.linalg.eigh(XX+SS)[0][0] < 0:
    break

In [ ]:
if 'Xopt' in locals() and 'Sopt' in locals():
    output_process(problem, params, XX, yy, SS, muRecs, Records, orders_used,
                   n, m, A, b, C, Xopt=Xopt, Sopt=Sopt)
else:
    output_process(problem, params, XX, yy, SS, muRecs, Records, orders_used,
                   n, m, A, b, C)


## Saving the Instance

In [ ]:
import scipy as sp
Gen = {'m':m, 'n':n, 'A': A, 'b':b, 'C':C, 'Xopt': Xopt, 'yopt':yopt, 'Sopt': Sopt, 'X_init': X_init, 'y_init': y_init, 'S_init': S_init}
sp.io.savemat('ND-Instance3.mat', Gen)

## CVXPY Solves

In [ ]:
import mosek
import os
os.environ["MOSEKLM_LICENSE_FILE"] = "..."

import cvxpy as cp
print(cp.installed_solvers())

In [ ]:
'''
Code used to call solvers such as MOSEK and SDPA to solve the isntances
'''
import cvxpy as cp

import re
from typing import List, Union, Tuple

def cp_solve(solver,n,m,A,b,C):
    X = cp.Variable((n+1,n+1), symmetric=True)

    constraints = [X >> 0]
    constraints += [
        cp.trace(A[i] @ X) == b[i] for i in range(m+1)
    ]
    prob = cp.Problem(cp.Minimize(cp.trace(C @ X)),
                    constraints)

    if solver == 'MOSEK':
        prob.solve(solver=cp.MOSEK, mosek_params={
            'MSK_DPAR_INTPNT_TOL_PFEAS': 1e-10,
            'MSK_DPAR_INTPNT_TOL_DFEAS': 1e-10,
            'MSK_DPAR_INTPNT_TOL_REL_GAP': 1e-10,
            'MSK_DPAR_INTPNT_CO_TOL_MU_RED': 1e-10,
            'MSK_DPAR_INTPNT_CO_TOL_REL_GAP': 1e-10,
            'MSK_DPAR_INTPNT_TOL_DSAFE': 10,
            'MSK_DPAR_INTPNT_TOL_PSAFE': 10
        }, verbose=True)

    elif solver == 'SDPA':
        prob.solve(solver=cp.SDPA, 
                   epsilonStar=1e-10, 
                   lambdaStar=1E-00, 
                   epsilonDash=1e-10, 
                   print='display', 
                   solver_verbose=1, verbose=1, 
                   resultFile='sdpa-log.txt', 
                   sdpaResult='result.txt')

    else:
        raise Exception('Solver not found')

'''
Following functions are defined to facilitate extracting gap
values from the log file of the solvers. 'extract_mu' is for
Mosek and 'extract_mu_sdpa' is for SDPA.
'''


def extract_mu(log_text: str) -> List[float]:
    """
    Extracts the MU column from MOSEK/CVXPY iteration tables inside arbitrary text.
    Works even if lines are prefixed with timestamps like:
      (CVXPY) Sep 01 06:26:38 PM: 0  1.6e+00 ...  1.0e+00  0.03
    """
    mu_values: List[float] = []
    lines = log_text.splitlines()

    # Numeric token (float or int, supports scientific notation)
    num_pat = re.compile(r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?')

    # Try to find the header first, then parse the rows after it
    header_idx = None
    for i, ln in enumerate(lines):
        if re.search(r'\bITE\b', ln) and re.search(r'\bMU\b', ln) and re.search(r'\bTIME\b', ln):
            header_idx = i
            break

    # Helper: return tokens from the part of the line after the last colon (to skip timestamps)
    def tokens_after_prefix(line: str):
        _, sep, tail = line.rpartition(':')  # split on the last colon only
        segment = tail if sep else line
        return num_pat.findall(segment)

    if header_idx is not None:
        for ln in lines[header_idx + 1:]:
            toks = tokens_after_prefix(ln)
            # Expect at least 9 numeric tokens: ITE PFEAS DFEAS GFEAS PRSTATUS POBJ DOBJ MU TIME
            if len(toks) >= 9:
                try:
                    mu_values.append(float(toks[7]))  # MU is the 8th numeric token
                except ValueError:
                    pass
        if mu_values:
            return mu_values

    # Fallback: pattern-match iteration rows directly (no header needed)
    row_pat = re.compile(
        r'^\s*(?:.*?:\s*)?'            # optional prefix "(...): "
        r'\d+\s+'                      # ITE
        r'[0-9.eE+\-]+\s+'             # PFEAS
        r'[0-9.eE+\-]+\s+'             # DFEAS
        r'[0-9.eE+\-]+\s+'             # GFEAS
        r'[0-9.eE+\-]+\s+'             # PRSTATUS
        r'[0-9.eE+\-]+\s+'             # POBJ
        r'[0-9.eE+\-]+\s+'             # DOBJ
        r'([0-9.eE+\-]+)\s+'           # MU (capture)
        r'[0-9.eE+\-]+\s*$',           # TIME
        re.M
    )
    return [float(x) for x in row_pat.findall(log_text)]


NumOrStr = Union[float, str]
OutType = Union[List[NumOrStr], List[Tuple[int, NumOrStr]]]

def extract_mu_sdpa(text: str, *, as_str: bool = False, fmt: str = ".2e",
                          include_index: bool = False) -> OutType:
    """
    Extract the 'mu' column from a CVXPY/SDPA iteration table such as:

       mu      thetaP  thetaD  objP  objD  alphaP  alphaD  beta
     0 1.0e+04 1.0e+00 ...

    Returns:
      - list of floats (default), or
      - list of strings with given format if as_str=True
      - optionally (index, value) pairs if include_index=True
    """
    # Float or int with optional exponent, allowing leading +/-
    num_pat = re.compile(r'[+\-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+\-]?\d+)?')

    lines = text.splitlines()

    # Find header line (tolerant to spacing/case)
    header_idx = None
    for i, ln in enumerate(lines):
        if (re.search(r'\bmu\b', ln, re.I) and
            re.search(r'\bthetaP\b', ln, re.I) and
            re.search(r'\bbeta\b', ln, re.I)):
            header_idx = i
            break

    start = header_idx + 1 if header_idx is not None else 0
    result: List[Union[float, str, Tuple[int, Union[float, str]]]] = []

    for ln in lines[start:]:
        if not ln.strip():
            continue
        # Ensure the line starts with an integer index
        m_idx = re.match(r'^\s*(\d+)\b', ln)
        if not m_idx:
            continue
        idx = int(m_idx.group(1))

        # Grab all numeric tokens; expect: [index, mu, thetaP, ...]
        toks = num_pat.findall(ln)
        if len(toks) < 2:
            continue

        # Second numeric token after the index is 'mu'
        try:
            mu_val = float(toks[1])
        except ValueError:
            continue

        out_val: NumOrStr = format(mu_val, fmt) if as_str else mu_val
        result.append((idx, out_val) if include_index else out_val)

    return result

In [ ]:
cp_solve('MOSEK',n,m,A,b,C)

In [ ]:
text = """
(CVXPY) Jun 14 01:49:57 PM: 0   1.4e+00  1.0e+00  1.0e+00  0.00e+00   -0.000000000e+00  -0.000000000e+00  1.0e+00  0.01  
(CVXPY) Jun 14 01:49:57 PM: 1   3.3e-01  2.3e-01  9.3e-02  9.86e-01   -1.619985511e-01  -1.100348097e-01  2.3e-01  0.02  
(CVXPY) Jun 14 01:49:57 PM: 2   4.3e-02  3.0e-02  4.2e-03  1.22e+00   -1.765740869e-01  -1.697203004e-01  3.0e-02  0.02  
(CVXPY) Jun 14 01:49:57 PM: 3   1.0e-02  7.2e-03  5.0e-04  1.03e+00   -1.431736613e-01  -1.419576519e-01  7.2e-03  0.02  
(CVXPY) Jun 14 01:49:57 PM: 4   3.6e-03  2.6e-03  1.3e-04  6.89e-01   -1.263543749e-01  -1.263829010e-01  2.6e-03  0.02  
(CVXPY) Jun 14 01:49:57 PM: 5   6.4e-04  4.5e-04  9.3e-06  9.97e-01   -1.260376311e-01  -1.259879529e-01  4.5e-04  0.02  
(CVXPY) Jun 14 01:49:57 PM: 6   7.8e-05  5.5e-05  3.7e-07  1.03e+00   -1.255374952e-01  -1.255274783e-01  5.5e-05  0.02  
(CVXPY) Jun 14 01:49:57 PM: 7   1.3e-05  8.9e-06  2.5e-08  1.02e+00   -1.254436638e-01  -1.254426860e-01  8.9e-06  0.02  
(CVXPY) Jun 14 01:49:57 PM: 8   2.0e-06  1.4e-06  1.6e-09  9.86e-01   -1.254376564e-01  -1.254375175e-01  1.4e-06  0.02  
(CVXPY) Jun 14 01:49:57 PM: 9   4.5e-07  3.2e-07  1.7e-10  9.93e-01   -1.254365102e-01  -1.254364810e-01  3.2e-07  0.02  
(CVXPY) Jun 14 01:49:57 PM: 10  1.3e-07  9.0e-08  2.6e-11  9.96e-01   -1.254362161e-01  -1.254362082e-01  9.0e-08  0.02  
(CVXPY) Jun 14 01:49:57 PM: 11  3.7e-08  2.6e-08  4.0e-12  9.99e-01   -1.254362059e-01  -1.254362037e-01  2.6e-08  0.02  
(CVXPY) Jun 14 01:49:57 PM: 12  8.7e-09  6.1e-09  4.6e-13  9.99e-01   -1.254361704e-01  -1.254361699e-01  6.1e-09  0.02  
(CVXPY) Jun 14 01:49:57 PM: 13  1.5e-09  1.2e-09  3.7e-14  1.00e+00   -1.254361693e-01  -1.254361692e-01  1.1e-09  0.02  
"""
mu_mosek = extract_mu(text)
print(mu_mosek)